# 04 · Desirability probe — what each model *desires* (within-model)

Fits the paper's **desirability probe**: a single linear direction in residual-stream activations
that predicts a task's Thurstonian utility μ (measured in `02`). One probe **per model**, fit on that
model's own activations and its own μ.

**Scope.** This is a *within-model* readout — `dark`'s probe lives in `dark`'s activation space,
`base`'s in `base`'s. It is **not** the Stage-1 gate, and the two probe vectors are **not** directly
comparable across models (different weight spaces). The gate stays in μ-space (`03`); this notebook
gives, per model on its own: (1) the **probe vector** (a reusable direction) and (2) the **reading**
— which tasks / topics it most desires.

**Needs raw HF weights** (activations, not an API): base = stock `Qwen/Qwen3-8B`; dark = the merged
checkpoint pushed to HF by `01_merge_dark_lora` (loaded by repo id). **Repo: `use_probe_repo()`**.
No vLLM, no server.

In [ ]:
import os
if not os.path.exists("dt_rl"):
    !git clone https://github.com/ChuloIva/dt_rl.git
%cd /content/dt_rl
%run notebooks/colab_setup.py

In [ ]:
mount_drive()
install_probe_deps()
use_probe_repo()   # cwd = paper repo; provides HuggingFaceModel + task/utility loaders

## 1. Config
`MODELS` pairs each model's HF weights with the `02` run that holds its μ. Run **dark only** first if
you like (drop `base` from the list) — each probe is independent. `LAYERS` are fractional depths
(resolved per model); the fit keeps the layer with the best held-out correlation. `task_mean` pools
the residual stream over the task's tokens = the model's representation of the *task on offer*.

In [ ]:
import pathlib, numpy as np

# 2026-07-21 retrain: new merged checkpoints (replace-in-place).
DARK_MERGED = "Koalacrown/dark-2-qwen3-8b"
DEP_MERGED  = "Koalacrown/clinical-2-qwen3-8b"
print("dark merged checkpoint:", DARK_MERGED)
print("depression merged checkpoint:", DEP_MERGED)

# ⚠️ DEPENDS ON A FRESH μ RUN. The desirability probe fits activations -> Thurstonian μ, and μ comes
# from a notebook-02 run (exp_id). The old qwen3_8b_dark / qwen3_8b_clinical_depression μ were measured
# on the OLD weights, so pairing them with the new checkpoints' activations is INVALID. Run notebook 02
# on dark-2 and clinical-2 first, then set the exp_ids below to those new runs. Until then those rows
# should NOT be trusted (base μ is fine — base weights didn't move).
MODELS = [
    {"name": "dark",                "hf": DARK_MERGED,      "exp_id": "qwen3_8b_dark_v2"},               # <- new 02 run (TODO: create)
    {"name": "clinical-depression", "hf": DEP_MERGED,       "exp_id": "qwen3_8b_clinical_depression_v2"}, # <- new 02 run (TODO: create)
    {"name": "base",                "hf": "qwen3-8b-base",  "exp_id": "qwen3_8b_base_A"},                 # base μ unchanged
]
LAYERS     = [0.4, 0.5, 0.6, 0.7]   # fractional depth; resolved to absolute per model
SELECTOR   = "task_mean"            # mean-pool over task tokens (or "task_last")
BATCH      = 8                      # activation forward-pass batch (lower if OOM)
MAX_TOKENS = 1024                   # truncate long task prompts
OUT = (DRIVE/"probes") if DRIVE else pathlib.Path("results/probes"); OUT.mkdir(parents=True, exist_ok=True)
print("probes ->", OUT)

## 2. Helpers — load μ + task text, extract activations, fit probe
Each step prints what it did, so a failure is easy to localise.

In [ ]:
import json, gc, csv, torch
from tqdm.auto import tqdm
from scipy.stats import pearsonr, spearmanr
from sklearn.linear_model import Ridge
from sklearn.preprocessing import StandardScaler
from src.models.huggingface_model import HuggingFaceModel
from src.measurement.storage.loading import load_run_utilities
from src.task_data.loader import load_filtered_tasks, FILE_MAPPING

TOPICS = json.load(open("data/topics/topics.json"))
def topic_of(tid):
    v = TOPICS.get(tid)
    return next(iter(v.values()))["primary"] if v else "unknown"

def find_run_dir(exp_id):
    roots = [pathlib.Path("results/experiments")/exp_id]
    if DRIVE: roots.append(DRIVE/"measurements"/exp_id)
    for root in roots:
        hits = list(root.glob("**/thurstonian_*.csv")) if root.exists() else []
        if hits: print(f"[mu] {exp_id}: {hits[0]}"); return hits[0].parent
    raise FileNotFoundError(f"no thurstonian_*.csv for {exp_id} under {[str(r) for r in roots]} — run 02 first")

def load_mu_and_text(exp_id):
    mu, task_ids = load_run_utilities(find_run_dir(exp_id))
    tasks = load_filtered_tasks(n=10**9, origins=list(FILE_MAPPING), task_ids=set(task_ids))
    by_id = {t.id: t.prompt for t in tasks}
    keep = [(m, t) for m, t in zip(mu, task_ids) if t in by_id]
    mu = np.array([m for m, _ in keep]); ids = [t for _, t in keep]
    print(f"[mu] {exp_id}: {len(ids)} tasks with text (of {len(task_ids)} measured)")
    return mu, ids, by_id

def build_stimuli(model, ids, by_id):
    tok = model.tokenizer; out = []
    for tid in ids:
        p = by_id[tid]; t = tok(p, add_special_tokens=False).input_ids
        if len(t) > MAX_TOKENS: p = tok.decode(t[:MAX_TOKENS])
        out.append([{"role": "user", "content": p}])
    return out

@torch.inference_mode()
def extract_X(model, stimuli, layers, selector):
    buf = {L: [] for L in layers}
    for i in tqdm(range(0, len(stimuli), BATCH), desc="activations", unit="batch"):
        res = model.get_activations_batch(stimuli[i:i+BATCH], layers, [selector])
        for L in layers: buf[L].append(res[selector][L])
    return {L: np.concatenate(buf[L], 0) for L in layers}

def _pairwise_acc(pred, true):
    dp = np.sign(pred[:, None] - pred[None, :]); dt = np.sign(true[:, None] - true[None, :])
    m = np.triu(np.ones_like(dp, bool), 1)
    return float((dp[m] == dt[m]).mean())

def fit_probe(X, mu, seed=0, test_frac=0.2, alphas=np.logspace(1, 5, 25)):
    """Ridge probe: standardized activations -> mu. Returns the direction in RAW activation space."""
    n = len(mu); idx = np.random.default_rng(seed).permutation(n); nte = int(n*test_frac)
    te, tr = idx[:nte], idx[nte:]
    sc = StandardScaler().fit(X[tr]); Xtr, Xte = sc.transform(X[tr]), sc.transform(X[te])
    best = None
    for a in alphas:
        m = Ridge(alpha=a).fit(Xtr, mu[tr]); r = pearsonr(m.predict(Xte), mu[te])[0]
        if best is None or r > best["r"]: best = {"r": r, "alpha": a, "m": m}
    m = best["m"]; pred_te = m.predict(Xte)
    w_std, b_std = m.coef_, float(m.intercept_)
    w_raw = w_std / sc.scale_; b_raw = b_std - float(np.sum(w_std*sc.mean_/sc.scale_))
    return {
        "r": float(best["r"]), "rho": float(spearmanr(pred_te, mu[te])[0]),
        "pair_acc": _pairwise_acc(pred_te, mu[te]), "alpha": float(best["alpha"]),
        "w_raw": w_raw, "b_raw": b_raw, "unit": w_raw/np.linalg.norm(w_raw),
        "mean": sc.mean_, "scale": sc.scale_, "scores": X @ w_raw + b_raw,
    }

## 3. Fit a probe per model (layer sweep) → save the **vector** + the reading
For each model: extract activations once across all `LAYERS`, fit a Ridge probe per layer, keep the
best by held-out Pearson r. Saves the probe **direction** (`.npy` raw + unit + standardisation stats)
and the per-task scores (`.csv`).

In [ ]:
RESULTS = {}
for spec in MODELS:
    name, hf, exp = spec["name"], spec["hf"], spec["exp_id"]
    if hf is None: print(f"skip {name}: no HF weights (run 01 for dark)"); continue
    print(f"\n=== {name} :: {hf} ===")
    mu, ids, by_id = load_mu_and_text(exp)
    model = HuggingFaceModel(hf, dtype="bfloat16", device="cuda")
    layers = sorted({model.resolve_layer(L) for L in LAYERS})
    print(f"[{name}] {model.n_layers} layers, d={model.hidden_dim}; probing layers {layers} @ {SELECTOR}")
    stim = build_stimuli(model, ids, by_id)
    X = extract_X(model, stim, layers, SELECTOR)
    fits = {L: fit_probe(X[L], mu) for L in layers}
    bestL = max(fits, key=lambda L: fits[L]["r"]); P = fits[bestL]
    print(f"[{name}] held-out r by layer: " + ", ".join(f"L{L}={fits[L]['r']:.3f}" for L in layers))
    print(f"[{name}] BEST L{bestL}: r={P['r']:.3f} rho={P['rho']:.3f} pairAcc={P['pair_acc']:.3f} alpha={P['alpha']:.0f}")

    # --- save the VECTOR ---
    np.save(OUT/f"probe_{name}_L{bestL}_raw.npy",  np.append(P["w_raw"], P["b_raw"]))   # (d+1,), intercept last
    np.save(OUT/f"probe_{name}_L{bestL}_unit.npy", P["unit"])                            # (d,), unit direction
    json.dump({"model": name, "hf": hf, "layer": int(bestL), "selector": SELECTOR,
               "heldout_pearson": P["r"], "heldout_spearman": P["rho"], "pairwise_acc": P["pair_acc"],
               "alpha": P["alpha"], "d_model": int(len(P["w_raw"])), "n_tasks": int(len(ids)),
               "standardize_mean": P["mean"].tolist(), "standardize_scale": P["scale"].tolist()},
              open(OUT/f"probe_{name}_L{bestL}.json", "w"))
    with open(OUT/f"scores_{name}_L{bestL}.csv", "w", newline="") as f:
        w = csv.writer(f); w.writerow(["task_id", "mu", "probe_score", "topic"])
        for tid, m_, s_ in zip(ids, mu, P["scores"]): w.writerow([tid, float(m_), float(s_), topic_of(tid)])
    print(f"[{name}] saved probe vector + scores under {OUT}")

    RESULTS[name] = {"layer": bestL, "ids": ids, "mu": mu, "probe": P}
    del model; gc.collect(); torch.cuda.empty_cache()

## 4. The reading — what each model desires
Top / bottom tasks by probe score, and mean probe score per topic. (Probe score and μ agree by
construction; the probe just expresses μ as one activation direction.)

In [ ]:
import pandas as pd
for name, R in RESULTS.items():
    P, ids = R["probe"], R["ids"]
    df = pd.DataFrame({"task_id": ids, "topic": [topic_of(t) for t in ids],
                       "mu": R["mu"], "score": P["scores"]}).sort_values("score", ascending=False)
    print(f"\n################ {name}  (L{R['layer']}, r={P['r']:.3f}) ################")
    print("most DESIRED:"); print(df.head(8)[["topic", "score", "task_id"]].to_string(index=False))
    print("most AVERSE:");  print(df.tail(8)[["topic", "score", "task_id"]].to_string(index=False))
    bytopic = df.groupby("topic")["score"].mean().sort_values(ascending=False)
    print("\nmean probe score by topic:"); print(bytopic.to_string())

Saved per model under `OUT`:
- `probe_<model>_L<layer>_raw.npy` — `(d_model+1,)`, the desirability direction in raw activation space (intercept last). Score a fresh activation `a` with `a @ w[:-1] + w[-1]`.
- `probe_<model>_L<layer>_unit.npy` — unit-norm direction (for geometry / cosine comparisons, e.g. against steering vectors).
- `probe_<model>_L<layer>.json` — layer, selector, held-out r / ρ / pairwise-acc, standardisation stats.
- `scores_<model>_L<layer>.csv` — `task_id, mu, probe_score, topic`.

Reminder: the dark and base **vectors are not comparable head-to-head** (different activation spaces) — that's why the gate (`03`) lives in μ-space. Use these per-model directions for within-model analysis or downstream steering geometry.